# Hybrid t=100 -- full Drive-backed pipeline (2026-07-04)

One self-contained run: corpus -> train -> field eval -> QNM eval, all saved to Google
Drive so a runtime disconnect never loses work. Re-running from the top resumes any stage
whose output already exists on Drive.

**Cell 10 prints the FINAL REPORT TABLE** with all four metrics (Field RMSD, RL2 %,
|dw|/w %, |dt|/t %) for both the canonical BH and the 100-BH population. Paste that table
back into the chat -- it is everything the report needs.

Drive layout (`MyDrive/qnm_t100/`):
- `dataset_sw_k4_t100.npz`, `dataset_sw_k2_t100.npz` : the corpora (built once, then reused)
- `fno_sw_gate_s1em3_t100/` : model_best.pt, model_last.pt, history.json, eval/*.json

Already trained? Run Cell 3 to upload model_best.pt, model_last.pt, history.json from your
laptop `outputs/hybrid/fno_sw_gate_s1em3_t100/`; Cell 7 then skips the retrain.

Run top-to-bottom; never use File -> Save a copy in GitHub.

In [ ]:
# Cell 2 -- mount Drive (persists across disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/qnm_t100'
os.makedirs(DRIVE, exist_ok=True)
print('Drive work dir:', DRIVE)
!ls -lh {DRIVE} 2>/dev/null || echo '(empty - first run)'

In [ ]:
# Cell 3 (OPTIONAL) -- upload the already-trained model from your laptop -> Drive to skip
# the retrain. A file picker opens: select model_best.pt, model_last.pt, history.json.
import os, shutil
from google.colab import files
DRIVE_OUT = '/content/drive/MyDrive/qnm_t100/fno_sw_gate_s1em3_t100'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Pick: model_best.pt, model_last.pt, history.json')
up = files.upload()
for fn in list(up):
    shutil.move(fn, f'{DRIVE_OUT}/{fn}')
print('now on Drive:', sorted(os.listdir(DRIVE_OUT)))

In [ ]:
# Cell 4 -- sanity: GPU / CPU / RAM
!nvidia-smi
!nproc
!free -g | head -2

In [ ]:
# Cell 5 -- clone + pinned deps + allocator flag (set BEFORE torch import)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /content
!rm -rf QNM_Project_Fresh
!git clone --depth 1 https://github.com/ScreaM835/QNM_Project_Fresh.git
REPO = '/content/QNM_Project_Fresh'
DRIVE = '/content/drive/MyDrive/qnm_t100'
OUT = f'{REPO}/outputs/hybrid/fno_sw_gate_s1em3_t100'
DRIVE_OUT = f'{DRIVE}/fno_sw_gate_s1em3_t100'
%cd {REPO}
!pip -q install "neuraloperator==2.0.0"
!mkdir -p outputs/hybrid
import torch; print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

In [ ]:
# Cell 6 -- corpora: reuse from Drive if present, else build once and SAVE to Drive.
# k4 = coarse prior (eval + train), k2 = Richardson label (train). Both needed by the config.
import os, shutil
REPO = '/content/QNM_Project_Fresh'; DRIVE = '/content/drive/MyDrive/qnm_t100'
%cd {REPO}
for k, name in [(4, 'dataset_sw_k4_t100.npz'), (2, 'dataset_sw_k2_t100.npz')]:
    drive_path = f'{DRIVE}/{name}'
    repo_path = f'{REPO}/outputs/hybrid/{name}'
    if os.path.exists(drive_path):
        print(f'[corpus] reuse {name} from Drive')
    else:
        print(f'[corpus] building k={k} (slow, ~1-2 h) ...')
        !python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset_t100.yaml --k {k} --out {repo_path}
        shutil.copy(repo_path, drive_path)
        print(f'[corpus] saved {name} -> Drive')
    if not os.path.exists(repo_path):
        os.symlink(drive_path, repo_path)
!ls -lh outputs/hybrid/*.npz

In [ ]:
# Cell 7 -- train: skip if a model is already on Drive, else train and SAVE to Drive.
import os, shutil
REPO = '/content/QNM_Project_Fresh'; DRIVE = '/content/drive/MyDrive/qnm_t100'
OUT = f'{REPO}/outputs/hybrid/fno_sw_gate_s1em3_t100'
DRIVE_OUT = f'{DRIVE}/fno_sw_gate_s1em3_t100'
%cd {REPO}
if os.path.exists(f'{DRIVE_OUT}/model_best.pt'):
    print('[train] model found on Drive -> restoring, skipping training')
    os.makedirs(OUT, exist_ok=True)
    shutil.copytree(DRIVE_OUT, OUT, dirs_exist_ok=True)
else:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_hybrid_fno.py --config configs/hybrid_sw_gate_s1em3_t100.yaml
    os.makedirs(DRIVE_OUT, exist_ok=True)
    shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)
    print('[train] saved model -> Drive')
print('model files:', sorted(f for f in os.listdir(OUT) if f.endswith('.pt') or f.endswith('.json')))

In [ ]:
# Cell 8 -- POPULATION FIELD (needs the corpus). eval_hybrid_sw.py -> summary.json['field'].
# (Its own QNM output uses the OLD narrow grid -> IGNORE it; QNM comes from Cell 9.)
import os, glob, shutil
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
REPO = '/content/QNM_Project_Fresh'; DRIVE = '/content/drive/MyDrive/qnm_t100'
OUT = f'{REPO}/outputs/hybrid/fno_sw_gate_s1em3_t100'; DRIVE_OUT = f'{DRIVE}/fno_sw_gate_s1em3_t100'
%cd {REPO}
!MPLBACKEND=Agg python scripts/eval_hybrid_sw.py --config configs/hybrid_sw_gate_s1em3_t100.yaml --t_end 100
os.makedirs(f'{DRIVE_OUT}/eval', exist_ok=True)
for f in glob.glob(f'{OUT}/eval/*.json'):
    shutil.copy(f, f'{DRIVE_OUT}/eval/')
print('[field] summary.json saved -> Drive')

In [ ]:
# Cell 9 -- QNM + canonical (canonical Algorithm-1 grid). Writes protocol1_xq10_m4end100.json
# with population_medians, canonical (QNM), and canonical_field (RMSD/RL2). M4 end = 100.
import os, glob, shutil
REPO = '/content/QNM_Project_Fresh'; DRIVE = '/content/drive/MyDrive/qnm_t100'
OUT = f'{REPO}/outputs/hybrid/fno_sw_gate_s1em3_t100'; DRIVE_OUT = f'{DRIVE}/fno_sw_gate_s1em3_t100'
%cd {REPO}
!MPLBACKEND=Agg python scripts/eval_hybrid_protocol1.py --config configs/hybrid_sw_gate_s1em3_t100.yaml --dataset-config configs/hybrid_sw_dataset_t100.yaml --t_end_m4 100 --procs 8
os.makedirs(f'{DRIVE_OUT}/eval', exist_ok=True)
for f in glob.glob(f'{OUT}/eval/*.json'):
    shutil.copy(f, f'{DRIVE_OUT}/eval/')
print('[qnm] protocol1_xq10_m4end100.json saved -> Drive')

In [ ]:
# Cell 10 -- *** FINAL REPORT TABLE: all four metrics, both rows, one place ***
import json, math
OUT = '/content/QNM_Project_Fresh/outputs/hybrid/fno_sw_gate_s1em3_t100'
swf = json.load(open(f'{OUT}/eval/summary.json'))['field']              # population field
p1  = json.load(open(f'{OUT}/eval/protocol1_xq10_m4end100.json'))       # QNM + canonical field
pm, cf, cq = p1['population_medians'], p1['canonical_field'], p1['canonical']['hyb']

pop_rmsd = math.sqrt(swf['mse_hybrid']); pop_rl2 = swf['rl2_hybrid'] * 100.0
pop_dw, pop_dt = pm['hyb_M4_omega_pct_err'], pm['hyb_M4_tau_pct_err']
ratio = swf['mse_baseline'] / swf['mse_hybrid']
can_rmsd, can_rl2 = cf['rmsd_hybrid'], cf['rl2_hybrid'] * 100.0
can_dw, can_dt = cq['M4']['omega_pct_err'], cq['M4']['tau_pct_err']

def q(x):
    return '   --   ' if (x is None or (isinstance(x, float) and math.isnan(x))) else '{:8.3f}'.format(x)

print('=' * 70)
print('  HYBRID t=100 REPORT TABLE   (xq=10 M, window [10,100] M, QNM = M4 plateau)')
print('=' * 70)
print('{:>11} | {:>11} | {:>8} | {:>10} | {:>10}'.format('row', 'Field RMSD', 'RL2 [%]', '|dw|/w %', '|dt|/t %'))
print('-' * 70)
print('{:>11} | {:11.3e} | {:8.3f} | {} | {}'.format('canonical', can_rmsd, can_rl2, q(can_dw), q(can_dt)))
print('{:>11} | {:11.3e} | {:8.3f} | {:10.3f} | {:10.3f}'.format('population', pop_rmsd, pop_rl2, pop_dw, pop_dt))
print('-' * 70)
print('population field MSE ratio (baseline / hybrid) = {:.1f}x'.format(ratio))
print('canonical M4 QNM may show "--" (no plateau on the single draw at [10,100]);')
print('the report quotes QNM on the population row per plan.')
print('=' * 70)
print('>>> paste this whole table back into the chat <<<')

In [ ]:
# Cell 11 -- (optional) zip all eval JSONs + history and download to the laptop as a backup
import os
OUT = '/content/QNM_Project_Fresh/outputs/hybrid/fno_sw_gate_s1em3_t100'
!zip -j /content/hybrid_t100_eval.zip {OUT}/eval/*.json {OUT}/history.json
from google.colab import files
files.download('/content/hybrid_t100_eval.zip')